# SDEV220 — Introducing Python: Chapters 11 and 16, Things to Do
**Student:** Cauy Whaley    
**Exercises:** 11.1, 11.2, 16.4, 16.5, and 16.8

## Exercise 11.1 — Create and import a module
Create a Python module named `zoo.py` containing an `hours()` function that prints `Open 9-5 daily`. Then import `zoo` and call the function.

**Explanation:** A module is a Python file that can be imported into another program. The first cell writes that file so the notebook can import it.

In [1]:
from pathlib import Path

Path("zoo.py").write_text(
    'def hours():\n    print("Open 9-5 daily")\n', encoding="utf-8"
)
print("Created zoo.py")

Created zoo.py


In [2]:
import zoo
zoo.hours()

Open 9-5 daily


## Exercise 11.2 — Import a module with an alias
Import the same `zoo` module as `menagerie` and call its `hours()` function.

**Explanation:** `as` gives a module a second name within the program. It does not change the original file.

In [3]:
import zoo as menagerie
menagerie.hours()

Open 9-5 daily


## Preparing the data for Exercise 16.5
Exercise 16.5 refers to `books2.csv`, created in the textbook's Exercise 16.3. We create it here to keep the notebook self-contained. The file contains the column headers `title`, `author`, and `year` and five records.

In [4]:
import csv

book_records = [
    ("The Weirdstone of Brisingamen", "Alan Garner", 1960),
    ("Perdido Street Station", "China Miéville", 2000),
    ("Thud!", "Terry Pratchett", 2005),
    ("The Spellman Files", "Lisa Lutz", 2007),
    ("Small Gods", "Terry Pratchett", 1992),
]

with open("books2.csv", "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["title", "author", "year"])
    writer.writerows(book_records)

print("Created books2.csv with", len(book_records), "book records")

Created books2.csv with 5 book records


## Exercise 16.4 — Create the SQLite database
Using Python's built-in `sqlite3` module, create `books.db` with a table called `books` and three columns:
- `title` — text
- `author` — text
- `year` — integer

**Explanation:** The connection creates the database file if it doesn't already exist. `CREATE TABLE IF NOT EXISTS` allows the notebook to be rerun without failing. To make the subsequent import repeatable, this exercise also clears previously imported rows.

In [5]:
import sqlite3

with sqlite3.connect("books.db") as connection:
    connection.execute("""
        CREATE TABLE IF NOT EXISTS books (
            title TEXT,
            author TEXT,
            year INTEGER
        )
    """)
    # Clear previous sample data so repeated notebook runs do not create duplicates.
    connection.execute("DELETE FROM books")

print("Created books.db and the books table")

Created books.db and the books table


## Exercise 16.5 — Read the CSV file and insert the records
Read `books2.csv`, then insert each book into the `books` table in `books.db`.

**Explanation:** `csv.DictReader` uses the first row for field names. The SQL statement uses `?` parameters, and `int()` makes the year an integer. `commit` is handled automatically by the connection context manager.

In [6]:
with open("books2.csv", newline="", encoding="utf-8") as csv_file:
    reader = csv.DictReader(csv_file)
    rows_to_insert = [
        (row["title"], row["author"], int(row["year"]))
        for row in reader
    ]

with sqlite3.connect("books.db") as connection:
    connection.executemany(
        "INSERT INTO books (title, author, year) VALUES (?, ?, ?)",
        rows_to_insert
    )
    count = connection.execute("SELECT COUNT(*) FROM books").fetchone()[0]

print("Books inserted:", count)
assert count == 5

Books inserted: 5


## Exercise 16.8 — Query SQLite with SQLAlchemy
Connect to the same `books.db` database using `sqlalchemy`, select the `title` column, and print titles in **alphabetical order**.

**Explanation:** `create_engine()` makes a database engine. The `text()` wrapper is the supported way to execute a SQL string in SQLAlchemy 2.x. `ORDER BY title ASC` sorts titles alphabetically.

In [7]:
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///books.db")
with engine.connect() as connection:
    results = connection.execute(
        text("SELECT title FROM books ORDER BY title ASC")
    )
    titles = [row[0] for row in results]

for title in titles:
    print(title)

assert titles == sorted(titles)
assert len(titles) == 5

Perdido Street Station
Small Gods
The Spellman Files
The Weirdstone of Brisingamen
Thud!
